In [14]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.nn.utils import parameters_to_vector, vector_to_parameters
import torch.nn.utils.prune as prune


from src.models import MLP
from src.data import get_mnist_loaders


RATIO = 0.99
MODEL = "MLP"
MODEL_CONFIG = {
            "input_size": 784,
            "hidden_sizes": [128],
            "num_classes": 10
        }
DATASET = "MNIST"
BATCH_SIZE = 128

SPARSE_FOLDER = f"saved_models/sparse/{MODEL}_{DATASET}_prune_ratio_{RATIO}"
PRUNE_FINETUNE_FOLDER = f"saved_models/prune_finetune/{MODEL}_{DATASET}_prune_ratio_{RATIO}"
DENSE_FOLDER = f"saved_models/dense/{MODEL}_{DATASET}"

DEVICE = "cpu"

MODEL_NAME_TO_CLASS = {
    "MLP": MLP,
}

In [15]:
@torch.no_grad()
def average_loss(
    model: torch.nn.Module,
    data_loader,
    criterion: torch.nn.Module,
    device: str | torch.device = "cpu",
    max_batches: int = 2,
) -> float:
    model = model.to(device)
    model.eval()

    total_loss = 0.0
    total_examples = 0
    for batch_idx, (data, target) in enumerate(data_loader):
        if batch_idx >= max_batches:
            break
        data = data.to(device)
        target = target.to(device)
        logits = model(data)
        batch_loss = criterion(logits, target)
        total_loss += float(batch_loss.item()) * data.size(0)
        total_examples += data.size(0)

    return total_loss / total_examples

In [22]:
data_loader

(<torch.utils.data.dataloader.DataLoader at 0x7f57c14cb9b0>,
 <torch.utils.data.dataloader.DataLoader at 0x7f571aca4980>)

In [ ]:
v1_mode = "sparse-init"
v2_mode = "finetune-init"

for sam in [True, False]:

    init_model = MODEL_NAME_TO_CLASS[MODEL](**MODEL_CONFIG)
    init_model.load_state_dict(torch.load(f"{DENSE_FOLDER}/{MODEL}_{DATASET}_sam_{sam}.pth", map_location=DEVICE))
    v1_model = MODEL_NAME_TO_CLASS[MODEL](**MODEL_CONFIG)
    # fake pruning
    parameters_to_prune = [
        (module, "weight")
        for _, module in v1_model.named_modules()
        if isinstance(module, (torch.nn.Linear, torch.nn.Conv2d))
    ]
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=0,
    )
    v1_model.load_state_dict(torch.load(f"{SPARSE_FOLDER}/{MODEL}_{DATASET}_sam_{sam}.pth", map_location=DEVICE))
    v2_model = MODEL_NAME_TO_CLASS[MODEL](**MODEL_CONFIG)
    # fake pruning
    parameters_to_prune = [
        (module, "weight")
        for _, module in v2_model.named_modules()
        if isinstance(module, (torch.nn.Linear, torch.nn.Conv2d))
    ]
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=0,
    )
    v2_model.load_state_dict(torch.load(f"{PRUNE_FINETUNE_FOLDER}/{MODEL}_{DATASET}_sam_train_{sam}_sam_finetune_{sam}.pth", map_location=DEVICE))

    origin = parameters_to_vector(init_model.parameters()).detach()
    v1 = parameters_to_vector(v1_model.parameters()).detach() - origin
    v2 = parameters_to_vector(v2_model.parameters()).detach() - origin

    # make v1 and v2 orthogonal
    v1_norm = torch.norm(v1)
    v2_norm = torch.norm(v2)
    basis_1 = v1 / v1_norm
    basis_2 = v2 - torch.dot(v1, v2) * basis_1
    v2 = basis_2 / torch.norm(basis_2)

    # data loader, criterion, and max_batches
    _, data_loader = get_mnist_loaders(batch_size=BATCH_SIZE)
    criterion = torch.nn.CrossEntropyLoss()
    max_batches = 100

    # define the grid params
    #x_limit = max(abs(sparse_target[0]), abs(finetune_target[0]), abs(dense_pruned_xy[0])) * 1.25
    #y_limit = max(abs(sparse_target[1]), abs(finetune_target[1]), abs(dense_pruned_xy[1])) * 1.25
    x_limit = 10.0
    y_limit = 10.0
    grid_size = 35
    x_coords = np.linspace(-x_limit, x_limit, grid_size)
    y_coords = np.linspace(-y_limit, y_limit, grid_size)

    # compute the loss landscape
    losses = np.zeros((len(y_coords), len(x_coords)), dtype=np.float64)
    for row, y_value in enumerate(y_coords):
        for col, x_value in enumerate(x_coords):
            model = MODEL_NAME_TO_CLASS[MODEL](**MODEL_CONFIG).to(DEVICE)

            vector = origin + float(x_value) * basis_1 + float(y_value) * basis_2
            vector_to_parameters(vector.clone(), model.parameters())
            losses[row, col] = average_loss(
                model=model,
                data_loader=data_loader,
                criterion=criterion,
                device=DEVICE,
                max_batches=max_batches,
            )
    
    # plot landscape
    sns.heatmap(losses)
    plt.show()
    

    

ValueError: too many values to unpack (expected 2)

In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import OrderedDict
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
from matplotlib import colors
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters

from src.data import get_mnist_loaders
from src.models import MLP


MODEL_NAME_TO_CLASS = {
    "MLP": MLP,
}

CHECKPOINT_EPOCH_RE = re.compile(r"epoch_(\d+)\.pth$")


def load_json(path: str | Path) -> dict:
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def build_model(model_name: str, model_params: dict) -> nn.Module:
    if model_name not in MODEL_NAME_TO_CLASS:
        raise ValueError(f"Unsupported model for notebook helper: {model_name}")
    return MODEL_NAME_TO_CLASS[model_name](**model_params)


def resolve_effective_state_dict(
    checkpoint_path: str | Path,
    reference_state_dict: OrderedDict[str, torch.Tensor],
) -> OrderedDict[str, torch.Tensor]:
    raw_state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if not isinstance(raw_state, (dict, OrderedDict)):
        raise TypeError(f"Unsupported checkpoint format at {checkpoint_path}: {type(raw_state)}")

    effective_state = OrderedDict()
    for key, ref_tensor in reference_state_dict.items():
        if key in raw_state:
            tensor = raw_state[key]
        elif key.endswith(".weight"):
            weight_orig_key = key.replace(".weight", ".weight_orig")
            weight_mask_key = key.replace(".weight", ".weight_mask")
            if weight_orig_key in raw_state and weight_mask_key in raw_state:
                tensor = raw_state[weight_orig_key] * raw_state[weight_mask_key]
            else:
                raise KeyError(f"Could not resolve parameter {key} in {checkpoint_path}")
        else:
            raise KeyError(f"Could not resolve parameter {key} in {checkpoint_path}")

        effective_state[key] = tensor.detach().clone().to(dtype=ref_tensor.dtype)

    return effective_state


def model_to_vector(model: nn.Module, state_dict: OrderedDict[str, torch.Tensor]) -> torch.Tensor:
    model_copy = build_model_from_reference(model)
    model_copy.load_state_dict(state_dict)
    return parameters_to_vector([param.detach().cpu() for param in model_copy.parameters()])


def build_model_from_reference(model: nn.Module) -> nn.Module:
    if isinstance(model, MLP):
        first_linear = model.linear_relu_stack[0]
        hidden_sizes = [layer.out_features for layer in model.linear_relu_stack if isinstance(layer, nn.Linear)]
        return MLP(
            input_size=first_linear.in_features,
            hidden_sizes=hidden_sizes,
            num_classes=model.output_layer.out_features,
        )
    raise ValueError(f"Unsupported model instance: {type(model)}")


def get_reference_model(config_path: str | Path) -> tuple[nn.Module, OrderedDict[str, torch.Tensor]]:
    config = load_json(config_path)
    model_name = config["model"]["name"]
    model_params = config["model"]["parameters"]
    model = build_model(model_name, model_params)
    return model, model.state_dict()


def effective_vector(
    checkpoint_path: str | Path,
    reference_model: nn.Module,
    reference_state_dict: OrderedDict[str, torch.Tensor],
) -> torch.Tensor:
    effective_state = resolve_effective_state_dict(checkpoint_path, reference_state_dict)
    model_copy = build_model_from_reference(reference_model)
    model_copy.load_state_dict(effective_state)
    return parameters_to_vector([param.detach().cpu() for param in model_copy.parameters()])


def project_vector(
    vector: torch.Tensor,
    origin: torch.Tensor,
    basis_u: torch.Tensor,
    basis_v: torch.Tensor,
) -> tuple[float, float]:
    offset = vector - origin
    return float(torch.dot(offset, basis_u)), float(torch.dot(offset, basis_v))


def orthonormal_basis(
    origin: torch.Tensor,
    sparse_vector: torch.Tensor,
    finetune_vector: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    direction_u = sparse_vector - origin
    direction_v = finetune_vector - origin

    basis_u = direction_u / torch.norm(direction_u)
    direction_v = direction_v - torch.dot(direction_v, basis_u) * basis_u
    basis_v = direction_v / torch.norm(direction_v)

    return basis_u, basis_v


def checkpoint_epoch(path: str | Path) -> int:
    match = CHECKPOINT_EPOCH_RE.search(str(path))
    if not match:
        raise ValueError(f"Could not parse epoch from checkpoint path: {path}")
    return int(match.group(1))


def sorted_checkpoint_paths(directory: str | Path) -> list[Path]:
    return sorted(Path(directory).glob("*.pth"), key=checkpoint_epoch)


def sparse_checkpoint_dir(root: str | Path, sparse_ratio: str, use_sam: bool) -> Path:
    return Path(root) / f"MLP_MNIST_prune_ratio_{sparse_ratio}" / "checkpoint"


def prune_finetune_checkpoint_dir(root: str | Path, finetune_ratio: str, use_sam: bool) -> Path:
    return (
        Path(root)
        / f"MLP_MNIST_prune_ratio_{finetune_ratio}"
        / "checkpoint"
        / f"sam_train_{use_sam}_sam_finetune_{use_sam}"
    )


def dense_checkpoint_path(root: str | Path, use_sam: bool) -> Path:
    return Path(root) / f"MLP_MNIST_sam_{use_sam}.pth"


def dense_checkpoint_dir(root: str | Path) -> Path:
    return Path(root) / "checkpoint"


def prune_dense_state_dict(
    dense_state_dict_path: str | Path,
    reference_state_dict: OrderedDict[str, torch.Tensor],
    prune_ratio: float,
) -> OrderedDict[str, torch.Tensor]:
    state_dict = resolve_effective_state_dict(dense_state_dict_path, reference_state_dict)

    weight_entries = [(key, tensor) for key, tensor in state_dict.items() if key.endswith(".weight")]
    flat_weights = torch.cat([tensor.reshape(-1).abs() for _, tensor in weight_entries])
    prune_count = int(math.floor(prune_ratio * flat_weights.numel()))

    if prune_count <= 0:
        return state_dict

    threshold = torch.kthvalue(flat_weights, prune_count).values.item()
    pruned_state = OrderedDict()
    for key, tensor in state_dict.items():
        if key.endswith(".weight"):
            mask = (tensor.abs() > threshold).to(dtype=tensor.dtype)
            pruned_state[key] = tensor * mask
        else:
            pruned_state[key] = tensor.clone()
    return pruned_state


def mnist_test_loader(batch_size: int = 256):
    _, test_loader = get_mnist_loaders(batch_size=batch_size)
    return test_loader


@torch.no_grad()
def average_loss(
    model: nn.Module,
    data_loader,
    criterion: nn.Module,
    device: str | torch.device = "cpu",
    max_batches: int = 2,
) -> float:
    model = model.to(device)
    model.eval()

    total_loss = 0.0
    total_examples = 0
    for batch_idx, (data, target) in enumerate(data_loader):
        if batch_idx >= max_batches:
            break
        data = data.to(device)
        target = target.to(device)
        logits = model(data)
        batch_loss = criterion(logits, target)
        total_loss += float(batch_loss.item()) * data.size(0)
        total_examples += data.size(0)

    return total_loss / total_examples


def loss_surface(
    reference_model: nn.Module,
    origin: torch.Tensor,
    basis_u: torch.Tensor,
    basis_v: torch.Tensor,
    x_coords: np.ndarray,
    y_coords: np.ndarray,
    data_loader,
    criterion: nn.Module,
    device: str | torch.device = "cpu",
    max_batches: int = 2,
) -> np.ndarray:
    losses = np.zeros((len(y_coords), len(x_coords)), dtype=np.float64)
    model = build_model_from_reference(reference_model)

    for row, y_value in enumerate(y_coords):
        for col, x_value in enumerate(x_coords):
            vector = origin + float(x_value) * basis_u + float(y_value) * basis_v
            vector_to_parameters(vector.clone(), model.parameters())
            losses[row, col] = average_loss(
                model=model,
                data_loader=data_loader,
                criterion=criterion,
                device=device,
                max_batches=max_batches,
            )

    return losses


def transform_surface_for_plot(surface: np.ndarray, mode: str = "raw", eps: float = 1e-8) -> tuple[np.ndarray, str]:
    if mode == "raw":
        return surface, "Average test loss"
    if mode == "log":
        return np.log10(np.maximum(surface, eps)), "log10(test loss)"
    if mode == "relative":
        min_value = float(surface.min())
        return surface - min_value, "test loss - min(surface)"
    raise ValueError(f"Unsupported surface transform mode: {mode}")


def trajectory_xy(
    checkpoint_paths: Iterable[str | Path],
    reference_model: nn.Module,
    reference_state_dict: OrderedDict[str, torch.Tensor],
    origin: torch.Tensor,
    basis_u: torch.Tensor,
    basis_v: torch.Tensor,
) -> tuple[np.ndarray, np.ndarray, list[int]]:
    xs = []
    ys = []
    epochs = []
    for path in checkpoint_paths:
        vector = effective_vector(path, reference_model, reference_state_dict)
        x_value, y_value = project_vector(vector, origin, basis_u, basis_v)
        xs.append(x_value)
        ys.append(y_value)
        epochs.append(checkpoint_epoch(path))
    return np.array(xs), np.array(ys), epochs


def interpolate_vectors(
    start: torch.Tensor,
    end: torch.Tensor,
    steps: int = 41,
) -> tuple[np.ndarray, list[torch.Tensor]]:
    alphas = np.linspace(0.0, 1.0, steps)
    vectors = [(1.0 - float(alpha)) * start + float(alpha) * end for alpha in alphas]
    return alphas, vectors


def vector_loss_series(
    vectors: Iterable[torch.Tensor],
    reference_model: nn.Module,
    data_loader,
    criterion: nn.Module,
    device: str | torch.device = "cpu",
    max_batches: int = 2,
) -> np.ndarray:
    model = build_model_from_reference(reference_model)
    losses = []
    for vector in vectors:
        vector_to_parameters(vector.clone(), model.parameters())
        losses.append(
            average_loss(
                model=model,
                data_loader=data_loader,
                criterion=criterion,
                device=device,
                max_batches=max_batches,
            )
        )
    return np.array(losses, dtype=np.float64)


def final_vectors_for_ratio(
    experiment: dict,
    use_sam: bool,
    config_path: str | Path = "configs/dense/MLP_MNIST_config.json",
    sparse_root: str | Path = "saved_models/sparse",
    prune_finetune_root: str | Path = "saved_models/prune_finetune_matched_sparse_ratios",
    dense_root: str | Path = "saved_models/dense/MLP_MNIST",
):
    reference_model, reference_state_dict = get_reference_model(config_path)
    origin_vector = effective_vector(Path(dense_root) / "MLP_MNIST_initial.pth", reference_model, reference_state_dict)
    dense_final_vector = effective_vector(dense_checkpoint_path(dense_root, use_sam), reference_model, reference_state_dict)
    sparse_final_vector = effective_vector(
        Path(sparse_root) / f"MLP_MNIST_prune_ratio_{experiment['sparse_ratio']}" / f"MLP_MNIST_sam_{use_sam}.pth",
        reference_model,
        reference_state_dict,
    )
    finetune_final_vector = effective_vector(
        Path(prune_finetune_root) / f"MLP_MNIST_prune_ratio_{experiment['finetune_ratio']}" / f"MLP_MNIST_sam_train_{use_sam}_sam_finetune_{use_sam}.pth",
        reference_model,
        reference_state_dict,
    )
    return reference_model, reference_state_dict, origin_vector, dense_final_vector, sparse_final_vector, finetune_final_vector


def ratio_plot(
    experiment: dict,
    config_path: str | Path = "configs/dense/MLP_MNIST_config.json",
    sparse_root: str | Path = "saved_models/sparse",
    prune_finetune_root: str | Path = "saved_models/prune_finetune_rerun_fast",
    dense_root: str | Path = "saved_models/dense/MLP_MNIST",
    grid_size: int = 35,
    max_batches: int = 2,
    device: str | torch.device = "cpu",
    color_clip_percentile: float = 90.0,
    surface_mode: str = "raw",
):
    reference_model, reference_state_dict = get_reference_model(config_path)
    origin_vector = effective_vector(Path(dense_root) / "MLP_MNIST_initial.pth", reference_model, reference_state_dict)

    criterion = nn.CrossEntropyLoss()
    test_loader = mnist_test_loader(batch_size=256)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

    optimizer_specs = [(False, "SGD"), (True, "SAM")]
    for axis, (use_sam, optimizer_label) in zip(axes, optimizer_specs):
        sparse_final_path = Path(sparse_root) / f"MLP_MNIST_prune_ratio_{experiment['sparse_ratio']}" / f"MLP_MNIST_sam_{use_sam}.pth"
        finetune_final_path = Path(prune_finetune_root) / f"MLP_MNIST_prune_ratio_{experiment['finetune_ratio']}" / f"MLP_MNIST_sam_train_{use_sam}_sam_finetune_{use_sam}.pth"

        sparse_final_vector = effective_vector(sparse_final_path, reference_model, reference_state_dict)
        finetune_final_vector = effective_vector(finetune_final_path, reference_model, reference_state_dict)
        basis_u, basis_v = orthonormal_basis(origin_vector, sparse_final_vector, finetune_final_vector)

        sparse_target = project_vector(sparse_final_vector, origin_vector, basis_u, basis_v)
        finetune_target = project_vector(finetune_final_vector, origin_vector, basis_u, basis_v)

        dense_pruned_state = prune_dense_state_dict(
            dense_state_dict_path=dense_checkpoint_path(dense_root, use_sam),
            reference_state_dict=reference_state_dict,
            prune_ratio=float(experiment["finetune_ratio"]),
        )
        dense_pruned_model = build_model_from_reference(reference_model)
        dense_pruned_model.load_state_dict(dense_pruned_state)
        dense_pruned_vector = parameters_to_vector([param.detach().cpu() for param in dense_pruned_model.parameters()])
        dense_pruned_xy = project_vector(dense_pruned_vector, origin_vector, basis_u, basis_v)

        x_limit = max(abs(sparse_target[0]), abs(finetune_target[0]), abs(dense_pruned_xy[0])) * 1.25
        y_limit = max(abs(sparse_target[1]), abs(finetune_target[1]), abs(dense_pruned_xy[1])) * 1.25
        x_limit = max(x_limit, 10.0)
        y_limit = max(y_limit, 10.0)
        x_coords = np.linspace(-x_limit, x_limit, grid_size)
        y_coords = np.linspace(-y_limit, y_limit, grid_size)

        surface = loss_surface(
            reference_model=reference_model,
            origin=origin_vector,
            basis_u=basis_u,
            basis_v=basis_v,
            x_coords=x_coords,
            y_coords=y_coords,
            data_loader=test_loader,
            criterion=criterion,
            device=device,
            max_batches=max_batches,
        )

        sparse_paths = sorted_checkpoint_paths(sparse_checkpoint_dir(sparse_root, experiment["sparse_ratio"], use_sam))
        prune_finetune_paths = sorted_checkpoint_paths(prune_finetune_checkpoint_dir(prune_finetune_root, experiment["finetune_ratio"], use_sam))
        dense_paths = sorted_checkpoint_paths(dense_checkpoint_dir(dense_root))
        dense_paths = [
            path for path in dense_paths
            if path.name.startswith(f"sam_{use_sam}_")
        ]

        sparse_x, sparse_y, sparse_epochs = trajectory_xy(
            sparse_paths,
            reference_model,
            reference_state_dict,
            origin_vector,
            basis_u,
            basis_v,
        )
        finetune_x, finetune_y, finetune_epochs = trajectory_xy(
            prune_finetune_paths,
            reference_model,
            reference_state_dict,
            origin_vector,
            basis_u,
            basis_v,
        )
        dense_x, dense_y, dense_epochs = trajectory_xy(
            dense_paths,
            reference_model,
            reference_state_dict,
            origin_vector,
            basis_u,
            basis_v,
        )

        surface_plot, colorbar_label = transform_surface_for_plot(surface, mode=surface_mode)
        vmin = float(surface_plot.min())
        vmax = float(np.percentile(surface_plot, color_clip_percentile))
        if vmax <= vmin:
            vmax = float(surface_plot.max())
        norm = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
        mesh = axis.contourf(x_coords, y_coords, surface_plot, levels=25, cmap="viridis", norm=norm)
        axis.contour(x_coords, y_coords, surface_plot, levels=12, colors="white", linewidths=0.4, alpha=0.45, norm=norm)
        axis.plot(dense_x, dense_y, "-o", color="#f8fafc", markersize=2.5, linewidth=1.0, alpha=0.95, label="Dense training")
        axis.plot(sparse_x, sparse_y, "-o", color="#f97316", markersize=3, linewidth=1.4, label="Sparse training")
        axis.plot(finetune_x, finetune_y, "-o", color="#38bdf8", markersize=3, linewidth=1.4, label="Prune-finetune")
        axis.scatter([0.0], [0.0], color="white", edgecolor="black", s=80, marker="*", zorder=6, label="w0")
        axis.scatter([dense_pruned_xy[0]], [dense_pruned_xy[1]], color="#fde047", edgecolor="black", s=60, marker="X", zorder=6, label="Pruned dense start")
        axis.scatter([dense_x[-1]], [dense_y[-1]], color="#cbd5e1", edgecolor="black", s=45, zorder=7)
        axis.scatter([sparse_x[-1]], [sparse_y[-1]], color="#ea580c", edgecolor="black", s=70, zorder=7)
        axis.scatter([finetune_x[-1]], [finetune_y[-1]], color="#0284c7", edgecolor="black", s=70, zorder=7)

        for index in (0, len(dense_epochs) - 1):
            axis.text(dense_x[index], dense_y[index], str(dense_epochs[index]), fontsize=8, color="white")
        for index in (0, len(sparse_epochs) - 1):
            axis.text(sparse_x[index], sparse_y[index], str(sparse_epochs[index]), fontsize=8, color="white")
        for index in (0, len(finetune_epochs) - 1):
            axis.text(finetune_x[index], finetune_y[index], str(finetune_epochs[index]), fontsize=8, color="white")

        axis.set_title(
            f"{optimizer_label} | sparse={experiment['sparse_ratio']} | prune-finetune={experiment['finetune_ratio']}"
        )
        axis.set_xlabel("alpha along (w_sparse - w0)")
        axis.set_ylabel("beta along (w_prune_finetune - w0)")
        axis.legend(loc="upper right", fontsize=8)

    fig.suptitle(experiment["title"], fontsize=14)
    colorbar = fig.colorbar(mesh, ax=axes, fraction=0.03, pad=0.02)
    colorbar.set_label(colorbar_label)
    return fig, axes


def common_ratio_plot(
    experiment: dict,
    basis_optimizer: bool = True,
    config_path: str | Path = "configs/dense/MLP_MNIST_config.json",
    sparse_root: str | Path = "saved_models/sparse",
    prune_finetune_root: str | Path = "saved_models/prune_finetune_matched_sparse_ratios",
    dense_root: str | Path = "saved_models/dense/MLP_MNIST",
    grid_size: int = 35,
    max_batches: int = 2,
    device: str | torch.device = "cpu",
    color_clip_percentile: float = 90.0,
    surface_mode: str = "raw",
):
    reference_model, reference_state_dict, origin_vector, _, sparse_basis_vector, finetune_basis_vector = final_vectors_for_ratio(
        experiment=experiment,
        use_sam=basis_optimizer,
        config_path=config_path,
        sparse_root=sparse_root,
        prune_finetune_root=prune_finetune_root,
        dense_root=dense_root,
    )
    basis_u, basis_v = orthonormal_basis(origin_vector, sparse_basis_vector, finetune_basis_vector)
    test_loader = mnist_test_loader(batch_size=256)
    criterion = nn.CrossEntropyLoss()

    all_points = [(0.0, 0.0)]
    trajectories = {}
    for use_sam, optimizer_label in ((False, "SGD"), (True, "SAM")):
        _, _, _, dense_final_vector, sparse_final_vector, finetune_final_vector = final_vectors_for_ratio(
            experiment=experiment,
            use_sam=use_sam,
            config_path=config_path,
            sparse_root=sparse_root,
            prune_finetune_root=prune_finetune_root,
            dense_root=dense_root,
        )
        dense_pruned_state = prune_dense_state_dict(
            dense_state_dict_path=dense_checkpoint_path(dense_root, use_sam),
            reference_state_dict=reference_state_dict,
            prune_ratio=float(experiment["finetune_ratio"]),
        )
        dense_pruned_model = build_model_from_reference(reference_model)
        dense_pruned_model.load_state_dict(dense_pruned_state)
        dense_pruned_vector = parameters_to_vector([param.detach().cpu() for param in dense_pruned_model.parameters()])

        dense_paths = [path for path in sorted_checkpoint_paths(dense_checkpoint_dir(dense_root)) if path.name.startswith(f"sam_{use_sam}_")]
        sparse_paths = sorted_checkpoint_paths(sparse_checkpoint_dir(sparse_root, experiment["sparse_ratio"], use_sam))
        finetune_paths = sorted_checkpoint_paths(prune_finetune_checkpoint_dir(prune_finetune_root, experiment["finetune_ratio"], use_sam))

        dense_xy = trajectory_xy(dense_paths, reference_model, reference_state_dict, origin_vector, basis_u, basis_v)
        sparse_xy = trajectory_xy(sparse_paths, reference_model, reference_state_dict, origin_vector, basis_u, basis_v)
        finetune_xy = trajectory_xy(finetune_paths, reference_model, reference_state_dict, origin_vector, basis_u, basis_v)
        dense_pruned_xy = project_vector(dense_pruned_vector, origin_vector, basis_u, basis_v)
        dense_final_xy = project_vector(dense_final_vector, origin_vector, basis_u, basis_v)
        sparse_final_xy = project_vector(sparse_final_vector, origin_vector, basis_u, basis_v)
        finetune_final_xy = project_vector(finetune_final_vector, origin_vector, basis_u, basis_v)

        trajectories[optimizer_label] = {
            "dense": dense_xy,
            "sparse": sparse_xy,
            "finetune": finetune_xy,
            "dense_pruned_xy": dense_pruned_xy,
            "dense_final_xy": dense_final_xy,
            "sparse_final_xy": sparse_final_xy,
            "finetune_final_xy": finetune_final_xy,
        }
        all_points.extend(
            list(zip(dense_xy[0], dense_xy[1])) + list(zip(sparse_xy[0], sparse_xy[1])) + list(zip(finetune_xy[0], finetune_xy[1])) +
            [dense_pruned_xy, dense_final_xy, sparse_final_xy, finetune_final_xy]
        )

    x_limit = max(abs(x) for x, _ in all_points) * 1.25 or 1.0
    y_limit = max(abs(y) for _, y in all_points) * 1.25 or 1.0
    x_limit = max(x_limit, 10.0)
    y_limit = max(y_limit, 10.0)
    x_coords = np.linspace(-x_limit, x_limit, grid_size)
    y_coords = np.linspace(-y_limit, y_limit, grid_size)
    surface = loss_surface(
        reference_model=reference_model,
        origin=origin_vector,
        basis_u=basis_u,
        basis_v=basis_v,
        x_coords=x_coords,
        y_coords=y_coords,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        max_batches=max_batches,
    )

    fig, axis = plt.subplots(1, 1, figsize=(10, 8), constrained_layout=True)
    surface_plot, colorbar_label = transform_surface_for_plot(surface, mode=surface_mode)
    vmin = float(surface_plot.min())
    vmax = float(np.percentile(surface_plot, color_clip_percentile))
    if vmax <= vmin:
        vmax = float(surface_plot.max())
    norm = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
    mesh = axis.contourf(x_coords, y_coords, surface_plot, levels=25, cmap="viridis", norm=norm)
    axis.contour(x_coords, y_coords, surface_plot, levels=12, colors="white", linewidths=0.4, alpha=0.45, norm=norm)
    palette = {
        "SGD": {"dense": "#e5e7eb", "sparse": "#f97316", "finetune": "#38bdf8"},
        "SAM": {"dense": "#94a3b8", "sparse": "#c2410c", "finetune": "#0369a1"},
    }
    for optimizer_label, data in trajectories.items():
        axis.plot(data["dense"][0], data["dense"][1], "-o", color=palette[optimizer_label]["dense"], markersize=2.5, linewidth=1.0, label=f"Dense {optimizer_label}")
        axis.plot(data["sparse"][0], data["sparse"][1], "-o", color=palette[optimizer_label]["sparse"], markersize=3, linewidth=1.4, label=f"Sparse {optimizer_label}")
        axis.plot(data["finetune"][0], data["finetune"][1], "-o", color=palette[optimizer_label]["finetune"], markersize=3, linewidth=1.4, label=f"Prune-finetune {optimizer_label}")
        axis.scatter([data["dense_pruned_xy"][0]], [data["dense_pruned_xy"][1]], color="#fde047", edgecolor="black", s=55, marker="X", zorder=6)

    axis.scatter([0.0], [0.0], color="white", edgecolor="black", s=90, marker="*", zorder=7, label="w0")
    basis_name = "SAM" if basis_optimizer else "SGD"
    axis.set_title(f"{experiment['title']} | common plane built from {basis_name} endpoints")
    axis.set_xlabel("alpha along common sparse direction")
    axis.set_ylabel("beta along common prune-finetune direction")
    axis.legend(loc="upper right", fontsize=8, ncol=2)
    colorbar = fig.colorbar(mesh, ax=axis, fraction=0.04, pad=0.02)
    colorbar.set_label(colorbar_label)
    return fig, axis


def barrier_plot(
    experiment: dict,
    config_path: str | Path = "configs/dense/MLP_MNIST_config.json",
    sparse_root: str | Path = "saved_models/sparse",
    prune_finetune_root: str | Path = "saved_models/prune_finetune_matched_sparse_ratios",
    dense_root: str | Path = "saved_models/dense/MLP_MNIST",
    steps: int = 41,
    max_batches: int = 2,
    device: str | torch.device = "cpu",
):
    reference_model, _, _, dense_sgd, sparse_sgd, finetune_sgd = final_vectors_for_ratio(
        experiment, False, config_path, sparse_root, prune_finetune_root, dense_root
    )
    _, _, _, dense_sam, sparse_sam, finetune_sam = final_vectors_for_ratio(
        experiment, True, config_path, sparse_root, prune_finetune_root, dense_root
    )
    criterion = nn.CrossEntropyLoss()
    test_loader = mnist_test_loader(batch_size=256)
    pairs = [
        ("Sparse SGD ↔ Prune-finetune SGD", sparse_sgd, finetune_sgd),
        ("Sparse SAM ↔ Prune-finetune SAM", sparse_sam, finetune_sam),
        ("Sparse SGD ↔ Sparse SAM", sparse_sgd, sparse_sam),
        ("Prune-finetune SGD ↔ Prune-finetune SAM", finetune_sgd, finetune_sam),
        ("Dense SGD ↔ Sparse SGD", dense_sgd, sparse_sgd),
        ("Dense SAM ↔ Sparse SAM", dense_sam, sparse_sam),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    for axis, (title, start, end) in zip(axes.ravel(), pairs):
        alphas, vectors = interpolate_vectors(start, end, steps=steps)
        losses = vector_loss_series(vectors, reference_model, test_loader, criterion, device=device, max_batches=max_batches)
        barrier = losses.max() - 0.5 * (losses[0] + losses[-1])
        axis.plot(alphas, losses, color="#2563eb", linewidth=1.8)
        axis.scatter([0.0, 1.0], [losses[0], losses[-1]], color="#0f172a", s=25)
        axis.set_title(f"{title}\nbarrier={barrier:.4f}", fontsize=10)
        axis.set_xlabel("interpolation coefficient")
        axis.set_ylabel("average test loss")

    fig.suptitle(f"Loss barriers | {experiment['title']}", fontsize=14)
    return fig, axes


def barrier_summary(
    experiment: dict,
    config_path: str | Path = "configs/dense/MLP_MNIST_config.json",
    sparse_root: str | Path = "saved_models/sparse",
    prune_finetune_root: str | Path = "saved_models/prune_finetune_matched_sparse_ratios",
    dense_root: str | Path = "saved_models/dense/MLP_MNIST",
    steps: int = 41,
    max_batches: int = 2,
    device: str | torch.device = "cpu",
) -> list[dict]:
    reference_model, _, _, dense_sgd, sparse_sgd, finetune_sgd = final_vectors_for_ratio(
        experiment, False, config_path, sparse_root, prune_finetune_root, dense_root
    )
    _, _, _, dense_sam, sparse_sam, finetune_sam = final_vectors_for_ratio(
        experiment, True, config_path, sparse_root, prune_finetune_root, dense_root
    )
    criterion = nn.CrossEntropyLoss()
    test_loader = mnist_test_loader(batch_size=256)
    pairs = [
        ("Sparse SGD ↔ Prune-finetune SGD", sparse_sgd, finetune_sgd),
        ("Sparse SAM ↔ Prune-finetune SAM", sparse_sam, finetune_sam),
        ("Sparse SGD ↔ Sparse SAM", sparse_sgd, sparse_sam),
        ("Prune-finetune SGD ↔ Prune-finetune SAM", finetune_sgd, finetune_sam),
        ("Dense SGD ↔ Sparse SGD", dense_sgd, sparse_sgd),
        ("Dense SAM ↔ Sparse SAM", dense_sam, sparse_sam),
    ]
    rows = []
    for title, start, end in pairs:
        alphas, vectors = interpolate_vectors(start, end, steps=steps)
        losses = vector_loss_series(vectors, reference_model, test_loader, criterion, device=device, max_batches=max_batches)
        rows.append(
            {
                "ratio": experiment["sparse_ratio"],
                "pair": title,
                "start_loss": float(losses[0]),
                "end_loss": float(losses[-1]),
                "max_loss": float(losses.max()),
                "barrier": float(losses.max() - 0.5 * (losses[0] + losses[-1])),
                "argmax_alpha": float(alphas[int(np.argmax(losses))]),
            }
        )
    return rows


In [ ]:
def run_ratio_views(experiments, *, surface_mode, color_clip_percentile, grid_size=35, max_batches=2, suffix=None):
    for experiment in experiments:
        fig, axes = ratio_plot(
            experiment=experiment,
            config_path=CONFIG_PATH,
            sparse_root=SPARSE_ROOT,
            prune_finetune_root=PRUNE_FINETUNE_ROOT,
            dense_root=DENSE_ROOT,
            grid_size=grid_size,
            max_batches=max_batches,
            color_clip_percentile=color_clip_percentile,
            surface_mode=surface_mode,
            device=DEVICE,
        )
        if SAVE_FIGURES and suffix is not None:
            file_name = f"ratio_{experiment['sparse_ratio']}_{suffix}.png"
            fig.savefig(OUTPUT_DIR / file_name, dpi=180, bbox_inches="tight")
        plt.show()


def run_common_views(experiments, *, basis_optimizer, surface_mode, color_clip_percentile, grid_size=35, max_batches=2, suffix=None):
    for experiment in experiments:
        fig, ax = common_ratio_plot(
            experiment=experiment,
            basis_optimizer=basis_optimizer,
            config_path=CONFIG_PATH,
            sparse_root=SPARSE_ROOT,
            prune_finetune_root=PRUNE_FINETUNE_ROOT,
            dense_root=DENSE_ROOT,
            grid_size=grid_size,
            max_batches=max_batches,
            color_clip_percentile=color_clip_percentile,
            surface_mode=surface_mode,
            device=DEVICE,
        )
        if SAVE_FIGURES and suffix is not None:
            file_name = f"common_{experiment['sparse_ratio']}_{suffix}.png"
            fig.savefig(OUTPUT_DIR / file_name, dpi=180, bbox_inches="tight")
        plt.show()


def run_barriers(experiments, *, steps=41, max_batches=2, suffix=None):
    for experiment in experiments:
        fig, axes = barrier_plot(
            experiment=experiment,
            config_path=CONFIG_PATH,
            sparse_root=SPARSE_ROOT,
            prune_finetune_root=PRUNE_FINETUNE_ROOT,
            dense_root=DENSE_ROOT,
            steps=steps,
            max_batches=max_batches,
            device=DEVICE,
        )
        if SAVE_FIGURES and suffix is not None:
            file_name = f"barrier_{experiment['sparse_ratio']}_{suffix}.png"
            fig.savefig(OUTPUT_DIR / file_name, dpi=180, bbox_inches="tight")
        plt.show()


In [ ]:
run_ratio_views(
    SELECTED_EXPERIMENTS,
    surface_mode="relative",
    color_clip_percentile=80,
    grid_size=35,
    max_batches=2,
    suffix="relative_clip80",
)
